# Kaggriculture: A/B two agents with error bars

**Use it in three steps.**

1. **Fork** this notebook.
2. Paste your agent into the cell marked **YOUR AGENT HERE** (bottom).
3. **Run all** -- you get a win rate with a 95% interval, and a note if the
   run was too short to tell your two agents apart.

That is the whole notebook: a measuring rig, not a strategy. It contains no
agent of mine and no finding about how to play. The worked examples use only
the three agents the environment already ships with -- `pass`, `random` and
`starter`.

**Why bother.** Two agents differing by a few percent need hundreds of
episodes before "it won more" means anything. Run twenty games, see 12--8,
and you have measured nothing: that result is what a coin does one time in
four. The rig below tells you when to believe your own comparison.

In [ ]:
import math
import statistics
from kaggle_environments import make

ENV = "kaggriculture"


def wilson(k, n, z=1.96):
    # 95% Wilson interval for k successes in n trials
    if n == 0:
        return 0.0, 1.0
    p = k / float(n)
    d = 1 + z * z / n
    c = p + z * z / (2 * n)
    s = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n))
    return max(0.0, (c - s) / d), min(1.0, (c + s) / d)


def duel(agent_a, agent_b, pairs=20, base_seed=1000, verbose=True):
    # Each seed is played TWICE with the seats swapped, so the seating
    # advantage cancels within the pair instead of averaging out over it.
    wins = losses = ties = 0
    margins = []
    for i in range(pairs):
        seed = base_seed + i
        for seat in (0, 1):
            order = [agent_a, agent_b] if seat == 0 else [agent_b, agent_a]
            env = make(ENV, configuration={"seed": seed})
            env.run(order)
            money = [f["money"] for f in env.state[0]["observation"]["farms"]]
            a, b = money[seat], money[1 - seat]
            margins.append(a - b)
            if a > b:
                wins += 1
            elif a < b:
                losses += 1
            else:
                ties += 1
    n = 2 * pairs
    dec = wins + losses                      # decisive episodes
    lo, hi = wilson(wins, n)                 # ties treated as non-wins
    dlo, dhi = wilson(wins, dec) if dec else (0.0, 1.0)
    drate = wins / float(dec) if dec else float("nan")
    out = dict(n=n, pairs=pairs, wins=wins, losses=losses, ties=ties,
               decisive=dec, rate=wins / float(n), lo=lo, hi=hi,
               decisive_rate=drate, dlo=dlo, dhi=dhi,
               median_margin=statistics.median(margins))
    if verbose:
        print("  {:>3} episodes ({} seeds x 2 seats)".format(n, pairs))
        print("  win {:<3} loss {:<3} tie {:<3}".format(wins, losses, ties))
        print("  share of all episodes won : {:6.1%}   95% CI [{:.1%}, {:.1%}]"
              .format(out["rate"], lo, hi))
        if dec:
            print("  share of DECISIVE won    : {:6.1%}   95% CI [{:.1%}, {:.1%}]   (n={})"
                  .format(drate, dlo, dhi, dec))
        print("  median money margin {:+,.0f}".format(out["median_margin"]))
        # The separation verdict uses decisive episodes. A tie is not evidence
        # for either agent, and counting it against A would report a drawn
        # match-up as a loss.
        if not dec:
            print("  -> every episode was a tie: no separation possible")
        elif dlo <= 0.5 <= dhi:
            print("  -> decisive-game interval covers 50%: does NOT separate them")
        else:
            print("  -> decisive-game interval excludes 50%")
        if ties and dec:
            print("  -> {}/{} episodes were ties ({:.0%}). The first rate is dragged"
                  .format(ties, n, ties / float(n)))
            print("     down by them; the decisive rate is the one to read.")
        elif ties:
            print("  -> all {} episodes were ties, so there is no decisive rate to"
                  .format(n))
            print("     read. Two agents that always draw cannot be ranked by wins;")
            print("     compare the money margin, or change the match-up.")
    return out

## How many episodes do you need?

Before running anything, this is arithmetic. The table below is the width of
the 95% Wilson interval around an observed 50% -- the worst case, and the
case you are usually in when two agents are close.

Read the last column as: *if my agent's true win rate were this, would this
many episodes notice?*

In [ ]:
print("{:>7}{:>11}{:>18}{:>30}".format(
    "seeds", "episodes", "95% CI at 50%", "smallest rate it separates"))
for pairs in (5, 10, 25, 50, 100, 250, 500):
    n = 2 * pairs
    lo, hi = wilson(n // 2, n)
    print("{:>7}{:>11}{:>18}{:>30}".format(
        pairs, n, "+/- {:.1%}".format((hi - lo) / 2), "{:.0%}".format(hi)))
print()
print("So 20 episodes cannot tell 50% from 70%, and 100 cannot tell 50% from")
print("60%. If you are comparing two versions of your own agent, assume you")
print("are in the close case and budget accordingly.")

## The design, in two sentences

**Paired seeds.** The same map is played by both arrangements, so the
map's own luck cancels inside the pair rather than being averaged away
across independent draws.

**Both seats.** Every seed is played once with your agent seated first and
once seated second, so any first-mover or seat asymmetry lands on both
agents equally instead of on whoever happened to be seated first.

Ties are counted and reported separately, because in this environment two
agents can finish on identical money -- and a rig that silently folds ties
into losses will tell you your agent is losing when it is drawing.

## Worked example: the three built-in agents

`pass`, `random` and `starter` ship with the environment, so this runs
without any agent of mine. The third match-up is a check on the rig itself:
an agent against a copy of itself has no edge, so its **decisive** games
should land near 50-50 and the interval should cover 50%.

In [ ]:
PAIRS = 20   # 40 episodes per match-up; about a minute each

print("starter vs random")
r1 = duel("starter", "random", pairs=PAIRS)
print()
print("random vs pass")
r2 = duel("random", "pass", pairs=PAIRS)
print()
print("random vs random   (the rig measuring itself)")
r3 = duel("random", "random", pairs=PAIRS)

The third block is the one to read first, and it is also the reason this
notebook reports two rates instead of one.

`random` against `random` produces **ties** -- both agents burn their money
and finish level. Counting ties as non-wins gives a win rate near zero,
which read alone would say the agent is losing badly to a copy of itself.
How many of the forty are decisive varies between runs, because the built-in
`random` agent is not pinned by the environment seed: locally this cell gave
3 wins, 3 losses and 34 ties, so the decisive rate was 50% with the interval
covering it, and on Kaggle it gave 40 ties and no decisive games at all. Both
readings are correct, and neither is "the agent is losing".

That is why the verdict line uses decisive episodes. A tie is not evidence
for either side, and a rig that charges it to one of them will report a
drawn match-up as a defeat.

## YOUR AGENT HERE

Replace the body of `my_agent` below. It takes the observation and returns
whatever the environment expects; the rig does not care what is inside it.

Then pick your opponent and your budget:

- comparing against a **built-in** agent -- `"starter"` is the usual bar
- comparing **two versions of your own** -- expect a small difference, so
  read the table above before choosing `pairs`

In [ ]:
def my_agent(obs):
    # ---- YOUR AGENT HERE -------------------------------------------------
    # Anything the environment accepts. This placeholder does nothing, so
    # out of the box it behaves like the built-in "pass" agent.
    return []
    # ----------------------------------------------------------------------


print("my_agent vs starter")
mine = duel(my_agent, "starter", pairs=10)

## What this does not do

- **It does not tell you *why* one agent won.** It counts episodes. A win
  rate with an interval is the end of what is measured here.
- **A result is about the seeds you ran.** `base_seed` is a parameter;
  changing it gives you a fresh sample, and if the answer moves a lot
  between blocks, that is information about your sample size rather than
  about your agent.
- **Ties are not wins, and they are not losses either.** Two rates are
  printed: the share of all episodes won, and the share of decisive ones.
  In a tie-heavy match-up the first understates the agent badly -- the
  worked example shows a self-play run reading 10% on it -- so the
  separation verdict is taken from the second.
- **Wall-clock cost is real.** One episode is roughly a second and a half in
  this environment, so 500 seeds is about half an hour of notebook time.
  That is the price of separating 50% from 54%.
- **The seed pins the map, not your agent.** Two runs of the third example
  above gave 4-3-33 and 3-3-34: the built-in `random` agent draws its own
  randomness, which the environment seed does not fix. If your agent has any
  internal randomness, seed it yourself or the pairing only cancels the map's
  luck and not your own.

---

*This notebook contains no agent, no strategy and no result about how to
play Kaggriculture. It is a measurement rig, published so that a comparison
between two agents can come with an error bar instead of a hunch.*